# InterPro API retrieval of Pfam domains for ProteinGym
Author: Sergio

The final files:

- Substitutions:
    - `pfam_overlapping_subs.csv`
    - `pfam_overlaping_subs_summary.csv`
    - `merged_subs_pfam.csv` (raw retrieval output, long format)
- Domainome:
    - `pfam_overlaping_domainome.csv`
    - `pfam_overlaping_subs_domainome.csv`
    - `merged_domainome_pfam.csv` (raw retrieval output, long format)
- Indels:
    - `pfam_indels.csv` (raw retrieval output, long format)
    - `pfam_summary_indels.csv`

You should be able to run the entire thing provided that you have `DMS_substitutions.csv`, `DMS_indels.csv` and `domainome_519_ref_file.csv`.

In [1]:
from urllib import request, error
import pandas as pd
import json
import requests

main function to retrieve Pfam information

In [4]:
# uniprot_ids = ["P69441", "P69905"]  # Example IDs
def get_pfam_from_interpro(assays, print_output=True):
    # first two strings are stored which are usually the uniprot ID
    uniprot_ids_2 = ['_'.join(dms_id.split('_')[0:2]) for dms_id in assays['DMS_id'].to_list()]
    print(len(uniprot_ids_2), "unique uniprot IDs in DMS assays")
    # the first name is also stored in case that is the only identifier
    uniprot_ids_1 = [(dms_id.split('_')[0]) for dms_id in assays['DMS_id'].to_list()]
    no_pfam_hits = []
    uniprot_pfams = []
    for i, uniprot_id in enumerate(uniprot_ids_2):
        url = f"https://rest.uniprot.org/uniprotkb/{uniprot_id}.json"
        res = requests.get(url).json()
        
        # define primary_accession
        primary_accession = res.get("primaryAccession")

        if primary_accession == None:
            url = f"https://rest.uniprot.org/uniprotkb/{uniprot_ids_1[i]}.json"
            res = requests.get(url).json()
            primary_accession = res.get("primaryAccession")
        # print(primary_accession)
        if primary_accession is None:
            print(f"Primary accession not found for {assays['DMS_id'][i]}, after trying {uniprot_id} or {uniprot_ids_1[i]}. Skipping.")
            uniprot_pfams.append({
                "DMS_id": assays['DMS_id'][i],
                "uniprot_id": None,
                "pfam_id": None,
                "start": None,
                "end": None
            })
            continue
        url = f"https://www.ebi.ac.uk/interpro/api/entry/pfam/protein/uniprot/{primary_accession}/"
        try:
            req = request.Request(url)
            with request.urlopen(req) as response:
                if response.status != 200:
                    print(f"Failed request: {response.status} for {primary_accession}")
                    uniprot_pfams.append({
                        "DMS_id": assays['DMS_id'][i],
                        "uniprot_id": None,
                        "pfam_id": None,
                        "start": None,
                        "end": None
                    })
                    continue
                raw = response.read().decode("utf-8")
                payload = json.loads(raw)
                # print(f"Data for {uniprot_id}: {payload}")
        except error.HTTPError as e:
            print(f"HTTP Error for {primary_accession}: {e.code} {e.reason}")
        except error.URLError as e:
            print(f"URL Error for {primary_accession}: {e.reason}")
        except json.JSONDecodeError:
            print(f"Failed to decode JSON for {primary_accession} — response might not be JSON.")

        pfam_match_count = 0
        for result in payload.get("results", []):
            metadata = result.get("metadata", {})
            if metadata.get("source_database", "").lower() == "pfam":
                pfam_id = metadata.get("accession")
                for protein in result.get("proteins", []):
                    for loc in protein.get("entry_protein_locations", []):
                        for fragment in loc.get("fragments", []):
                            pfam_match_count += 1
                            pfam_dict = {
                                "DMS_id": assays['DMS_id'][i],
                                "uniprot_id": primary_accession,
                                "pfam_id": pfam_id,
                                "start": fragment.get("start"),
                                "end": fragment.get("end")
                            }
                            uniprot_pfams.append(pfam_dict)     
        
        if print_output:
            print(f"There were {pfam_match_count} Pfam entries found for {primary_accession}.")
            if pfam_match_count == 0:
                print(f"No Pfam entries found for {primary_accession}.")
                no_pfam_hits.append(primary_accession)

                        ##########
        if i % 10 == 0:
            print(f"Processed {i} UniProt IDs")
    
    if len(no_pfam_hits) > 0 and print_output:
        print("No Pfam hits found for the following UniProt IDs:")
        for uniprot_id in no_pfam_hits:
            print(f" - {uniprot_id}")
    else:
        print("All UniProt IDs had Pfam hits. Yay!")
    return uniprot_pfams



In [ ]:
def check_target_seq(target_seq, pfam_seq):
    pass

## Start with substitutions from ProteinGym

In [40]:
subs_assays = pd.read_csv("DMS_substitutions.csv")
uniprot_pfams = get_pfam_from_interpro(subs_assays, print_output=False)

FileNotFoundError: [Errno 2] No such file or directory: 'DMS_substitutions.csv'

In [4]:
df_pfam_subs = pd.DataFrame(uniprot_pfams)
df_pfam_subs

,DMS_id,uniprot_id,pfam_id,start,end
0,A0A140D2T1_ZIKV_Sourisseau_2019,A0A140D2T1,PF00869,292.0,592.0
1,A0A140D2T1_ZIKV_Sourisseau_2019,A0A140D2T1,PF00948,797.0,1148.0
2,A0A140D2T1_ZIKV_Sourisseau_2019,A0A140D2T1,PF00949,1519.0,1669.0
3,A0A140D2T1_ZIKV_Sourisseau_2019,A0A140D2T1,PF00972,2772.0,3222.0
4,A0A140D2T1_ZIKV_Sourisseau_2019,A0A140D2T1,PF01002,1378.0,1502.0
...,...,...,...,...,...
671,YAIA_ECOLI_Tsuboyama_2023_2KVT,P0AAN5,PF16362,1.0,62.0
672,YAP1_HUMAN_Araya_2012,P46937,PF00397,173.0,202.0
673,YAP1_HUMAN_Araya_2012,P46937,PF00397,232.0,261.0
674,YAP1_HUMAN_Araya_2012,P46937,PF15238,84.0,99.0


In [5]:
# let's add the mutation start and end positions
region_df = subs_assays['region_mutated'].str.extract(r'(?P<start>\d+)-(?P<end>\d+)').astype(int)
subs_assays['mutation_start'] = region_df['start']
subs_assays['mutation_end'] = region_df['end']
subs_assays

,DMS_index,DMS_id,DMS_filename,UniProt_ID,taxon,source_organism,target_seq,seq_len,includes_multiple_mutants,DMS_total_number_mutants,...,raw_DMS_directionality,raw_DMS_mutant_column,weight_file_name,pdb_file,pdb_range,ProteinGym_version,raw_mut_offset,coarse_selection_type,mutation_start,mutation_end
0,DMS_sub_0,A0A140D2T1_ZIKV_Sourisseau_2019,A0A140D2T1_ZIKV_Sourisseau_2019.csv,A0A140D2T1_ZIKV,Virus,Zika virus (ZIKV),MKNPKKKSGGFRIVNMLKRGVARVNPLGGLKRLPAGLLLGHGPIRM...,3423,False,9576,...,1,mutant,A0A140D2T1_ZIKV_theta_0.01.npy,A0A140D2T1_ZIKV.pdb,291-794,0.1,NaN,OrganismalFitness,291,794
1,DMS_sub_1,A0A192B1T2_9HIV1_Haddox_2018,A0A192B1T2_9HIV1_Haddox_2018.csv,A0A192B1T2_9HIV1,Virus,Human immunodeficiency virus 1,MRVKGIQMNSQHLLRWGIMILGMIMICSVAGNLWVTVYYGVPVWKD...,852,False,12577,...,1,mutant,A0A192B1T2_9HIV1_theta_0.01.npy,A0A192B1T2_9HIV1.pdb,1-852,0.1,NaN,OrganismalFitness,30,691
2,DMS_sub_2,A0A1I9GEU1_NEIME_Kennouche_2019,A0A1I9GEU1_NEIME_Kennouche_2019.csv,A0A1I9GEU1_NEIME,Prokaryote,Neisseria meningitidis,FTLIELMIVIAIVGILAAVALPAYQDYTARAQVSEAILLAEGQKSA...,161,False,922,...,1,mutants,A0A1I9GEU1_NEIME_theta_0.2.npy,A0A1I9GEU1_NEIME.pdb,1-161,0.1,NaN,Activity,1,161
3,DMS_sub_3,A0A247D711_LISMN_Stadelmann_2021,A0A247D711_LISMN_Stadelmann_2021.csv,A0A247D711_LISMN,Prokaryote,Listeria monocytogenes,MNINDLIREIKNKDYTVKLSGTDSNSITQLIIRVNNDGNEYVISES...,87,False,1653,...,1,mutant,A0A247D711_LISMN_b03_theta_0.2.npy,A0A247D711_LISMN.pdb,1-87,1.0,NaN,Activity,1,87
4,DMS_sub_4,A0A2Z5U3Z0_9INFA_Doud_2016,A0A2Z5U3Z0_9INFA_Doud_2016.csv,A0A2Z5U3Z0_9INFA,Virus,Influenza A virus (A/WSN/1933(H1N1)),MKAKLLVLLYAFVATDADTICIGYHANNSTDTVDTILEKNVAVTHS...,565,False,10715,...,1,mutant,A0A2Z5U3Z0_9INFA_theta_0.01.npy,A0A2Z5U3Z0_9INFA.pdb,1-565,0.1,NaN,OrganismalFitness,2,565
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
212,DMS_sub_212,VKOR1_HUMAN_Chiasson_2020_activity,VKOR1_HUMAN_Chiasson_2020_activity.csv,VKOR1_HUMAN,Human,Homo sapiens,MGSTWGSPGWVRLALCLTGLVLSLYALHVKAARARDRDYRALCDVG...,163,False,697,...,1,variant,VKOR1_HUMAN_theta_0.2.npy,VKOR1_HUMAN.pdb,1-163,0.1,NaN,Activity,3,163
213,DMS_sub_213,VRPI_BPT7_Tsuboyama_2023_2WNM,VRPI_BPT7_Tsuboyama_2023_2WNM.csv,VRPI_BPT7,Virus,Escherichia phage T7 (Bacteriophage T7),SLSVDNKKFWATVESSEHSFEVPIYAETLDEALELAEWQYVPAGFE...,56,False,1047,...,1,mut_type,VRPI_BPT7_theta0.01_2023-08-07_b02.npy,VRPI_BPT7.pdb,1-56,1.0,NaN,Stability,1,56
214,DMS_sub_214,YAIA_ECOLI_Tsuboyama_2023_2KVT,YAIA_ECOLI_Tsuboyama_2023_2KVT.csv,YAIA_ECOLI,Prokaryote,Escherichia coli,PREAYIVTIEKGKPGQTVTWYQLRADHPKPDSLISEHPTAQEAMDA...,52,True,1890,...,1,mut_type,YAIA_ECOLI_theta0.2_2023-08-07_b03.npy,YAIA_ECOLI.pdb,1-52,1.0,NaN,Stability,1,52
215,DMS_sub_215,YAP1_HUMAN_Araya_2012,YAP1_HUMAN_Araya_2012.csv,YAP1_HUMAN,Human,Homo sapiens,MDPGQQPPPQPAPQGQGQPPSQPPQGQGPPSGPGQPAPAATQAAPQ...,504,True,10075,...,1,mutant,YAP1_HUMAN_theta_0.2.npy,YAP1_HUMAN.pdb,1-504,0.1,NaN,Binding,170,203


In [6]:
subs_assays.to_csv("DMS_substitutions_with_ranges.csv", index=False)

let's look for overlapping regions

In [8]:
df_pfam_subs.shape, subs_assays.shape

((676, 5), (217, 48))

In [ ]:
df_pfam_subs.isnull().any(axis=1).sum() # two that could not be matched

np.int64(2)

In [12]:
# Merge df_pfam_subs with subs_assays to get mutation_start and mutation_end for each DMS_id
merged_subs = df_pfam_subs.merge(
    subs_assays[['DMS_id', 'mutation_start', 'mutation_end']],
    on='DMS_id',
    how='left'
)

# Find Pfam domains that overlap with the mutated region
overlapping = merged_subs[
    (merged_subs['start'] <= merged_subs['mutation_end']) &
    (merged_subs['end'] >= merged_subs['mutation_start'])
].copy()

# Calculate percentage of Pfam domain that is within the mutated range
def percent_domain_within_range(row):
    if pd.isnull(row['start']) or pd.isnull(row['end']):
        return None
    overlap_start = max(row['start'], row['mutation_start'])
    overlap_end = min(row['end'], row['mutation_end'])
    overlap = max(0, overlap_end - overlap_start + 1)
    domain_length = row['end'] - row['start'] + 1
    return overlap / domain_length if domain_length > 0 else 0

overlapping['proportion_domain_mutated'] = overlapping.apply(percent_domain_within_range, axis=1)
overlapping['proportion_domain_mutated'] = overlapping['proportion_domain_mutated'].round(3)
overlapping

,DMS_id,uniprot_id,pfam_id,start,end,mutation_start,mutation_end,proportion_domain_mutated
0,A0A140D2T1_ZIKV_Sourisseau_2019,A0A140D2T1,PF00869,292.0,592.0,291,794,1.000
12,A0A140D2T1_ZIKV_Sourisseau_2019,A0A140D2T1,PF02832,594.0,692.0,291,794,1.000
16,A0A140D2T1_ZIKV_Sourisseau_2019,A0A140D2T1,PF21659,695.0,789.0,291,794,1.000
17,A0A192B1T2_9HIV1_Haddox_2018,A0A192B1T2,PF00516,33.0,139.0,30,691,1.000
18,A0A192B1T2_9HIV1_Haddox_2018,A0A192B1T2,PF00516,142.0,500.0,30,691,1.000
...,...,...,...,...,...,...,...,...
669,VKOR1_HUMAN_Chiasson_2020_activity,Q9BQB6,PF07884,10.0,149.0,3,163,1.000
670,VRPI_BPT7_Tsuboyama_2023_2WNM,P03704,PF16857,14.0,60.0,1,56,0.915
671,YAIA_ECOLI_Tsuboyama_2023_2KVT,P0AAN5,PF16362,1.0,62.0,1,52,0.839
672,YAP1_HUMAN_Araya_2012,P46937,PF00397,173.0,202.0,170,203,1.000


In [13]:
overlapping['DMS_id'].nunique(), subs_assays['DMS_id'].nunique()

(193, 217)

there are a few assays that do not have pfam domains overlapping with the mutated region

In [14]:
merged_subs

,DMS_id,uniprot_id,pfam_id,start,end,mutation_start,mutation_end
0,A0A140D2T1_ZIKV_Sourisseau_2019,A0A140D2T1,PF00869,292.0,592.0,291,794
1,A0A140D2T1_ZIKV_Sourisseau_2019,A0A140D2T1,PF00948,797.0,1148.0,291,794
2,A0A140D2T1_ZIKV_Sourisseau_2019,A0A140D2T1,PF00949,1519.0,1669.0,291,794
3,A0A140D2T1_ZIKV_Sourisseau_2019,A0A140D2T1,PF00972,2772.0,3222.0,291,794
4,A0A140D2T1_ZIKV_Sourisseau_2019,A0A140D2T1,PF01002,1378.0,1502.0,291,794
...,...,...,...,...,...,...,...
671,YAIA_ECOLI_Tsuboyama_2023_2KVT,P0AAN5,PF16362,1.0,62.0,1,52
672,YAP1_HUMAN_Araya_2012,P46937,PF00397,173.0,202.0,170,203
673,YAP1_HUMAN_Araya_2012,P46937,PF00397,232.0,261.0,170,203
674,YAP1_HUMAN_Araya_2012,P46937,PF15238,84.0,99.0,170,203


In [ ]:
merged_subs.to_csv("merged_subs_pfam.csv", index=False) # this is the merged file with all the information

In [17]:
overlapping.to_csv("pfam_overlapping_subs.csv", index=False)

In [18]:
pfam_summary = (
    overlapping.groupby('DMS_id')['pfam_id']
    .agg(lambda x: ','.join(sorted(set(x))))
    .reset_index()
    .rename(columns={'pfam_id': 'pfam_ids'})
)
# Add uniprot_id by taking the first uniprot_id for each DMS_id in overlapping
pfam_summary['uniprot_id'] = overlapping.groupby('DMS_id')['uniprot_id'].first().reindex(pfam_summary['DMS_id']).values
pfam_summary['pfam_count'] = pfam_summary['pfam_ids'].apply(lambda x: len(x.split(',')) if pd.notnull(x) and x != '' else 0)
pfam_summary = pfam_summary[['DMS_id', 'uniprot_id', 'pfam_ids', 'pfam_count']]
pfam_summary

,DMS_id,uniprot_id,pfam_ids,pfam_count
0,A0A140D2T1_ZIKV_Sourisseau_2019,A0A140D2T1,"PF00869,PF02832,PF21659",3
1,A0A192B1T2_9HIV1_Haddox_2018,A0A192B1T2,"PF00516,PF00517",2
2,A0A1I9GEU1_NEIME_Kennouche_2019,A0A1I9GEU1,"PF00114,PF07963",2
3,A0A247D711_LISMN_Stadelmann_2021,A0A247D711,PF24304,1
4,A0A2Z5U3Z0_9INFA_Doud_2016,A0A2Z5U3Z0,PF00509,1
...,...,...,...,...
188,VKOR1_HUMAN_Chiasson_2020_activity,Q9BQB6,PF07884,1
189,VRPI_BPT7_Tsuboyama_2023_2WNM,P03704,PF16857,1
190,YAIA_ECOLI_Tsuboyama_2023_2KVT,P0AAN5,PF16362,1
191,YAP1_HUMAN_Araya_2012,P46937,PF00397,1


In [19]:
overlapping['DMS_id'].nunique(), merged_subs['DMS_id'].nunique()

(193, 217)

In [20]:
pfam_summary.to_csv("pfam_overlapping_subs_summary.csv", index=False)

## let's do the domainome

In [5]:
domainome = pd.read_csv("/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym2/EVCouplings/output/Beltran_Lehner_2025/temp_ref_file.csv")
uniprot_pfams_domainome = get_pfam_from_interpro(domainome, print_output=False)
df_pfam_domainome = pd.DataFrame(uniprot_pfams_domainome)
df_pfam_domainome

519 unique uniprot IDs in DMS assays
Failed request: 204 for A0A2R8Y422
Processed 10 UniProt IDs
Processed 20 UniProt IDs
Processed 30 UniProt IDs
Processed 40 UniProt IDs
Processed 50 UniProt IDs
Processed 60 UniProt IDs
Processed 70 UniProt IDs
Processed 80 UniProt IDs
Processed 90 UniProt IDs
Processed 100 UniProt IDs
Processed 110 UniProt IDs
Processed 120 UniProt IDs
Processed 130 UniProt IDs
Processed 140 UniProt IDs
Processed 150 UniProt IDs
Processed 160 UniProt IDs
Processed 170 UniProt IDs
Processed 180 UniProt IDs
Processed 190 UniProt IDs
Processed 200 UniProt IDs
Processed 210 UniProt IDs
Processed 220 UniProt IDs
Processed 230 UniProt IDs
Processed 240 UniProt IDs
Processed 250 UniProt IDs
Processed 260 UniProt IDs
Processed 270 UniProt IDs
Processed 280 UniProt IDs
Processed 290 UniProt IDs
Processed 300 UniProt IDs
Failed request: 204 for Q3SY89
Processed 310 UniProt IDs
Processed 320 UniProt IDs
Processed 330 UniProt IDs
Processed 340 UniProt IDs
Processed 350 UniProt 

,DMS_id,uniprot_id,pfam_id,start,end
0,A0A2R8Y422_HUMAN_Beltran_2025_PF00240_2,None,None,NaN,NaN
1,FEZF1_HUMAN_Beltran_2025_PF00096_289,A0PJY2,PF00096,260.0,279.0
2,FEZF1_HUMAN_Beltran_2025_PF00096_289,A0PJY2,PF00096,288.0,310.0
3,FEZF1_HUMAN_Beltran_2025_PF00096_289,A0PJY2,PF00096,316.0,338.0
4,FEZF1_HUMAN_Beltran_2025_PF00096_289,A0PJY2,PF00096,344.0,366.0
...,...,...,...,...,...
2500,PCLO_HUMAN_Beltran_2025_PF05715_1058,Q9Y6V0,PF00168,4707.0,4825.0
2501,PCLO_HUMAN_Beltran_2025_PF05715_1058,Q9Y6V0,PF00168,5025.0,5134.0
2502,PCLO_HUMAN_Beltran_2025_PF05715_1058,Q9Y6V0,PF00595,4508.0,4583.0
2503,PCLO_HUMAN_Beltran_2025_PF05715_1058,Q9Y6V0,PF05715,588.0,646.0


In [7]:
df_pfam_domainome[df_pfam_domainome['DMS_id']=='CALL4_HUMAN_Beltran_2025_PF13499_48']

,DMS_id,uniprot_id,pfam_id,start,end
2020,CALL4_HUMAN_Beltran_2025_PF13499_48,Q96GE6,PF13499,133.0,188.0


In [42]:
domainome['mutation_start'] = domainome['offset']
domainome['mutation_end'] = domainome['offset'] + domainome['seq_len'] - 1
domainome

,DMS_id,MSA_bitscore,MSA_num_seqs,MSA_perc_cov,MSA_num_cov,MSA_N_eff,MSA_Neff_L,MSA_Neff_L_category,weight_file_name,MSA_filename,...,MSA_len,MSA_theta,taxon,source_organism,includes_multiple_mutants,DMS_wildtype_score,first_author,coarse_selection_type,mutation_start,mutation_end
0,A0A2R8Y422_HUMAN_Beltran_2025_PF00240_2,0.7,61559,0.971,68,5929.1,84.701140,medium,A0A2R8Y422_HUMAN_Beltran_2025_PF00240_2_theta_...,A0A2R8Y422_HUMAN_Beltran_2025_PF00240_2_theta_...,...,70,0.2,Human,Saccharomyces cerevisiae,False,0,Beltran,Abundance,2,71
1,FEZF1_HUMAN_Beltran_2025_PF00096_289,1.5,645870,0.955,21,1813.5,82.430408,medium,FEZF1_HUMAN_Beltran_2025_PF00096_289_theta_0.8...,FEZF1_HUMAN_Beltran_2025_PF00096_289_theta_0.8...,...,22,0.2,Human,Saccharomyces cerevisiae,False,0,Beltran,Abundance,289,310
2,SPD2B_HUMAN_Beltran_2025_PF00018_155,0.9,33216,0.945,52,721.9,13.125128,medium,SPD2B_HUMAN_Beltran_2025_PF00018_155_theta_0.8...,SPD2B_HUMAN_Beltran_2025_PF00018_155_theta_0.8...,...,55,0.2,Human,Saccharomyces cerevisiae,False,0,Beltran,Abundance,155,209
3,SPD2B_HUMAN_Beltran_2025_PF00018_222,0.9,45592,0.898,53,947.7,16.062838,medium,SPD2B_HUMAN_Beltran_2025_PF00018_222_theta_0.8...,SPD2B_HUMAN_Beltran_2025_PF00018_222_theta_0.8...,...,59,0.2,Human,Saccharomyces cerevisiae,False,0,Beltran,Abundance,222,280
4,A2RRE5_HUMAN_Beltran_2025_PF01846_267,0.7,4543,0.952,60,128.7,2.042930,medium,A2RRE5_HUMAN_Beltran_2025_PF01846_267_theta_0....,A2RRE5_HUMAN_Beltran_2025_PF01846_267_theta_0....,...,63,0.2,Human,Saccharomyces cerevisiae,False,0,Beltran,Abundance,267,329
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
514,CD2AP_HUMAN_Beltran_2025_PF14604_1,1.1,91191,0.881,52,1947.7,33.012110,medium,CD2AP_HUMAN_Beltran_2025_PF14604_1_theta_0.8_b...,CD2AP_HUMAN_Beltran_2025_PF14604_1_theta_0.8_b...,...,59,0.2,Human,Saccharomyces cerevisiae,False,0,Beltran,Abundance,1,59
515,CD2AP_HUMAN_Beltran_2025_PF14604_109,1.1,101933,0.869,53,1764.6,28.927503,medium,CD2AP_HUMAN_Beltran_2025_PF14604_109_theta_0.8...,CD2AP_HUMAN_Beltran_2025_PF14604_109_theta_0.8...,...,61,0.2,Human,Saccharomyces cerevisiae,False,0,Beltran,Abundance,109,169
516,NCOR2_HUMAN_Beltran_2025_PF00249_613,0.7,57518,0.828,48,1467.9,25.307762,medium,NCOR2_HUMAN_Beltran_2025_PF00249_613_theta_0.8...,NCOR2_HUMAN_Beltran_2025_PF00249_613_theta_0.8...,...,58,0.2,Human,Saccharomyces cerevisiae,False,0,Beltran,Abundance,613,670
517,USH1C_HUMAN_Beltran_2025_PF00595_207,0.7,123337,0.831,74,1477.5,16.601235,medium,USH1C_HUMAN_Beltran_2025_PF00595_207_theta_0.8...,USH1C_HUMAN_Beltran_2025_PF00595_207_theta_0.8...,...,89,0.2,Human,Saccharomyces cerevisiae,False,0,Beltran,Abundance,207,295


In [43]:
domainome.to_csv("domainome_with_ranges.csv", index=False)

In [5]:
df_pfam_domainome.isnull().any(axis=1).sum() # the two that could not be retrieved

2

In [11]:
df_pfam_domainome[df_pfam_domainome.isnull().any(axis=1) == True]

,DMS_id,uniprot_id,pfam_id,start,end
0,A0A2R8Y422_HUMAN_Beltran_2025_PF00240_2,None,None,NaN,NaN
1597,ELB3B_HUMAN_Beltran_2025_PF08711_3,None,None,NaN,NaN


In [44]:
# do this for domainome now
# Merge df_pfam_domainome with subs_assays to get mutation_start and mutation_end for each DMS_id
merged_domainome = df_pfam_domainome.merge(
    domainome[['DMS_id', 'mutation_start', 'mutation_end']],
    on='DMS_id',
    how='left'
)

# Find Pfam domains that overlap with the mutated region
overlapping = merged_domainome[
    (merged_domainome['start'] <= merged_domainome['mutation_end']) &
    (merged_domainome['end'] >= merged_domainome['mutation_start'])
].copy()

# Calculate percentage of Pfam domain that is within the mutated range
def percent_domain_within_range(row):
    if pd.isnull(row['start']) or pd.isnull(row['end']):
        return None
    overlap_start = max(row['start'], row['mutation_start'])
    overlap_end = min(row['end'], row['mutation_end'])
    overlap = max(0, overlap_end - overlap_start + 1)
    domain_length = row['end'] - row['start'] + 1
    return overlap / domain_length if domain_length > 0 else 0

overlapping['proportion_domain_mutated'] = overlapping.apply(percent_domain_within_range, axis=1)
overlapping['proportion_domain_mutated'] = overlapping['proportion_domain_mutated'].round(3)
# get rid of overhanging residues overlapping with other neighboring domains
# overlapping = overlapping[overlapping['proportion_domain_mutated'] > 0.5] # because 0.5 is min frag size filter in MSA

In [14]:
merged_domainome

,DMS_id,uniprot_id,pfam_id,start,end,mutation_start,mutation_end
0,A0A2R8Y422_HUMAN_Beltran_2025_PF00240_2,None,None,NaN,NaN,2,71
1,FEZF1_HUMAN_Beltran_2025_PF00096_289,A0PJY2,PF00096,260.0,279.0,289,310
2,FEZF1_HUMAN_Beltran_2025_PF00096_289,A0PJY2,PF00096,288.0,310.0,289,310
3,FEZF1_HUMAN_Beltran_2025_PF00096_289,A0PJY2,PF00096,316.0,338.0,289,310
4,FEZF1_HUMAN_Beltran_2025_PF00096_289,A0PJY2,PF00096,344.0,366.0,289,310
...,...,...,...,...,...,...,...
2500,PCLO_HUMAN_Beltran_2025_PF05715_1058,Q9Y6V0,PF00168,4707.0,4825.0,1058,1114
2501,PCLO_HUMAN_Beltran_2025_PF05715_1058,Q9Y6V0,PF00168,5025.0,5134.0,1058,1114
2502,PCLO_HUMAN_Beltran_2025_PF05715_1058,Q9Y6V0,PF00595,4508.0,4583.0,1058,1114
2503,PCLO_HUMAN_Beltran_2025_PF05715_1058,Q9Y6V0,PF05715,588.0,646.0,1058,1114


Some domains have a few amino acids overhang within the target sequence, so should be ok

1 domain `SNF5_HUMAN_Beltran_2025_PF04855_181` only sequenced 35.1% of the domain. Should probably remove

In [15]:
merged_domainome.to_csv("merged_domainome_pfam.csv", index=False) # this is the merged file with all the information

In [2]:
merged_domainome = pd.read_csv("merged_domainome_pfam.csv")

In [3]:
merged_domainome[merged_domainome['DMS_id']=='CALL4_HUMAN_Beltran_2025_PF13499_48']

,DMS_id,uniprot_id,pfam_id,start,end,mutation_start,mutation_end
2020,CALL4_HUMAN_Beltran_2025_PF13499_48,Q96GE6,PF13499,133.0,188.0,48,123


In [45]:
overlapping.shape, domainome.shape

((520, 8), (519, 29))

In [26]:
overlapping['DMS_id'].value_counts()[overlapping['DMS_id'].value_counts() > 1]

DMS_id
S100G_BOVIN_Beltran_2025_PF01023_1    2
Name: count, dtype: int64

`S100G_BOVIN_Beltran_2025_PF01023_1` covers two full domains:  PF00036, PF01023

so there are 1 domain with 2 pfam hits --> should probably separate the scoring files into two

In [27]:
overlapping['DMS_id'].nunique(), domainome['DMS_id'].nunique()

(508, 519)

In [28]:
set(domainome['DMS_id']) - set(overlapping['DMS_id'])

{'A0A2R8Y422_HUMAN_Beltran_2025_PF00240_2',
 'CALL4_HUMAN_Beltran_2025_PF13499_48',
 'ELB3B_HUMAN_Beltran_2025_PF08711_3',
 'HOIL1_HUMAN_Beltran_2025_PF01485_369',
 'M18BP_HUMAN_Beltran_2025_PF00249_878',
 'NEBU_HUMAN_Beltran_2025_PF14604_6609',
 'OVOL2_HUMAN_Beltran_2025_PF00096_215',
 'PO6F1_HUMAN_Beltran_2025_PF00157_139',
 'PRD15_HUMAN_Beltran_2025_PF00096_1085',
 'SNF5_HUMAN_Beltran_2025_PF04855_181',
 'ZMYM6_HUMAN_Beltran_2025_PF06467_427'}

there are some that are not in the overlaps - this is because there is no overlapping Pfam domain! See example below. These also include the two that were not searchable (`ELB3B_HUMAN_Beltran_2025_PF08711_3` and `A0A2R8Y422_HUMAN_Beltran_2025_PF00240_2`), low domain coverage (`SNF5_HUMAN_Beltran_2025_PF04855_181`).

In [36]:
domainome[domainome['DMS_id'] == 'CALL4_HUMAN_Beltran_2025_PF13499_48']

,DMS_id,MSA_bitscore,MSA_num_seqs,MSA_perc_cov,MSA_num_cov,MSA_N_eff,MSA_Neff_L,MSA_Neff_L_category,weight_file_name,MSA_filename,EVCouplings_model_filename,target_seq,DMS_filename,offset,seq_len,pdb_file,mutation_start,mutation_end
400,CALL4_HUMAN_Beltran_2025_PF13499_48,0.9,54187,0.895,68,4934.2,64.923361,medium,CALL4_HUMAN_Beltran_2025_PF13499_48_theta_0.8_...,CALL4_HUMAN_Beltran_2025_PF13499_48_theta_0.8_...,CALL4_HUMAN_Beltran_2025_PF13499_48_b0.9.model,LSQDQINEYKECFSLYDKQQRGKIKATDLMVAMRCLGASPTPGEVQ...,CALL4_HUMAN_Beltran_2025_PF13499_48.csv,48,76,CALL4_HUMAN_Beltran_2025_PF13499_48.pdb,48,123


In [37]:
df_pfam_domainome[df_pfam_domainome['DMS_id'] == 'CALL4_HUMAN_Beltran_2025_PF13499_48']

,DMS_id,uniprot_id,pfam_id,start,end
2022,CALL4_HUMAN_Beltran_2025_PF13499_48,Q96GE6,PF13499,133.0,188.0


Check sequence similarity between the target sequence of these and uniprot sequence at these regions

In [38]:
overlapping.to_csv("pfam_overlapping_domainome.csv", index=False)

In [22]:
pfam_summary = (
    overlapping.groupby('DMS_id')['pfam_id']
    .agg(lambda x: ','.join(sorted(set(x))))
    .reset_index()
    .rename(columns={'pfam_id': 'pfam_ids'})
)
# Add uniprot_id by taking the first uniprot_id for each DMS_id in overlapping
pfam_summary['uniprot_id'] = overlapping.groupby('DMS_id')['uniprot_id'].first().reindex(pfam_summary['DMS_id']).values
pfam_summary['pfam_count'] = pfam_summary['pfam_ids'].apply(lambda x: len(x.split(',')) if pd.notnull(x) and x != '' else 0)
pfam_summary = pfam_summary[['DMS_id', 'uniprot_id', 'pfam_ids', 'pfam_count']]
pfam_summary

,DMS_id,uniprot_id,pfam_ids,pfam_count
0,A2RRE5_HUMAN_Beltran_2025_PF01846_267,A2RRE5,PF16512,1
1,ABL1_HUMAN_Beltran_2025_PF00018_64,P00519,PF00018,1
2,ABLM3_HUMAN_Beltran_2025_PF00412_149,O94929,PF00412,1
3,ABLM3_HUMAN_Beltran_2025_PF00412_210,O94929,PF00412,1
4,ADNP_HUMAN_Beltran_2025_PF00046_770,Q9H2P0,PF00046,1
...,...,...,...,...
505,ZO2_HUMAN_Beltran_2025_PF00595_509,Q9UDY2,PF00595,1
506,ZRAB2_HUMAN_Beltran_2025_PF00641_11,O95218,PF00641,1
507,ZRAB2_HUMAN_Beltran_2025_PF00641_9,O95218,PF00641,1
508,ZYX_HUMAN_Beltran_2025_PF00412_504,Q15942,PF00412,1


In [23]:
pfam_summary.to_csv("pfam_overlapping_domainome_summary.csv", index=False)

## finally let's do indels

In [58]:
indels_dms = pd.read_csv("DMS_indels.csv")

In [59]:
uniprot_pfams_indels = get_pfam_from_interpro(indels_dms, print_output=False)
df_pfam_indels = pd.DataFrame(uniprot_pfams_indels)
df_pfam_indels


66 unique uniprot IDs in DMS assays
Processed 0 UniProt IDs
Processed 10 UniProt IDs
Processed 20 UniProt IDs
Processed 30 UniProt IDs
Processed 40 UniProt IDs
Processed 50 UniProt IDs
Processed 60 UniProt IDs
All UniProt IDs had Pfam hits. Yay!


,DMS_id,uniprot_id,pfam_id,start,end
0,A4_HUMAN_Seuma_2022_indels,P05067,PF00014,290,341
1,A4_HUMAN_Seuma_2022_indels,P05067,PF02177,31,131
2,A4_HUMAN_Seuma_2022_indels,P05067,PF03494,676,713
3,A4_HUMAN_Seuma_2022_indels,P05067,PF10515,716,766
4,A4_HUMAN_Seuma_2022_indels,P05067,PF12924,132,188
...,...,...,...,...,...
262,VILI_CHICK_Tsuboyama_2023_1YU5_indels,P02640,PF00626,532,595
263,VILI_CHICK_Tsuboyama_2023_1YU5_indels,P02640,PF00626,634,707
264,VILI_CHICK_Tsuboyama_2023_1YU5_indels,P02640,PF02209,791,826
265,VRPI_BPT7_Tsuboyama_2023_2WNM_indels,P03704,PF16857,14,60


In [60]:
import pandas as pd

df_pfam_indels = pd.DataFrame(uniprot_pfams_indels)
df_pfam_indels

,DMS_id,uniprot_id,pfam_id,start,end
0,A4_HUMAN_Seuma_2022_indels,P05067,PF00014,290,341
1,A4_HUMAN_Seuma_2022_indels,P05067,PF02177,31,131
2,A4_HUMAN_Seuma_2022_indels,P05067,PF03494,676,713
3,A4_HUMAN_Seuma_2022_indels,P05067,PF10515,716,766
4,A4_HUMAN_Seuma_2022_indels,P05067,PF12924,132,188
...,...,...,...,...,...
262,VILI_CHICK_Tsuboyama_2023_1YU5_indels,P02640,PF00626,532,595
263,VILI_CHICK_Tsuboyama_2023_1YU5_indels,P02640,PF00626,634,707
264,VILI_CHICK_Tsuboyama_2023_1YU5_indels,P02640,PF02209,791,826
265,VRPI_BPT7_Tsuboyama_2023_2WNM_indels,P03704,PF16857,14,60


In [65]:
df_pfam_indels.to_csv("pfam_indels.csv", index=False)

to make the summary file, there are is no overlapping to be done, so the we just count the pfam hits from the pfam_indels file directly

In [62]:
df_pfam_summary = (
    df_pfam_indels.groupby('DMS_id')['pfam_id']
    .agg(['unique', 'count'])
    .rename(columns={'unique': 'pfam_ids', 'count': 'pfam_count'})
    .reset_index()
)
df_pfam_summary['pfam_ids'] = df_pfam_summary['pfam_ids'].apply(lambda x: ','.join(sorted(set(filter(pd.notnull, x)))))
df_pfam_summary

,DMS_id,pfam_ids,pfam_count
0,A4_HUMAN_Seuma_2022_indels,"PF00014,PF02177,PF03494,PF10515,PF12924,PF12925",6
1,AMFR_HUMAN_Tsuboyama_2023_4G3O_indels,"PF02845,PF13639,PF18442,PF25563",4
2,ARGR_ECOLI_Tsuboyama_2023_1AOY_indels,"PF01316,PF02863",2
3,B1LPA6_ECOSM_Russ_2020_indels,"PF00800,PF01817",2
4,BBC1_YEAST_Tsuboyama_2023_1TG0_indels,"PF00018,PF25459",2
...,...,...,...
61,UBR5_HUMAN_Tsuboyama_2023_1I2T_indels,"PF00632,PF00658,PF11547",3
62,VG08_BPP22_Tsuboyama_2023_2GP8_indels,PF09306,1
63,VILI_CHICK_Tsuboyama_2023_1YU5_indels,"PF00626,PF02209",7
64,VRPI_BPT7_Tsuboyama_2023_2WNM_indels,PF16857,1


In [63]:
df_pfam_summary.to_csv("pfam_summary_indels.csv", index=False)